# 6.2.3 Reinforcement Learning from Human Feedback (RLHF)

Demonstrate reward model training with Bradley-Terry pairwise preferences and the PPO KL-penalised objective: reward − β · KL(π ‖ π_ref).

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)

def bt_prob(r_w, r_l):
    return 1.0 / (1.0 + np.exp(r_l - r_w))

n_resp = 200
true_r = rng.normal(0, 1, n_resp)
est_r = np.zeros(n_resp)
n_pairs = 400
w_idx = rng.integers(0, n_resp, n_pairs)
l_idx = rng.integers(0, n_resp, n_pairs)
l_idx[w_idx == l_idx] = (l_idx[w_idx == l_idx] + 1) % n_resp
pref = true_r[w_idx] > true_r[l_idx]

for _ in range(200):
    for i in range(n_pairs):
        p = bt_prob(est_r[w_idx[i]], est_r[l_idx[i]])
        g = 1 - p if pref[i] else -p
        est_r[w_idx[i]] += 0.01 * g
        est_r[l_idx[i]] -= 0.01 * g

corr = np.corrcoef(true_r, est_r)[0, 1]
print(f'Reward model correlation: {corr:.3f}')

In [ ]:
betas = np.linspace(0, 2, 100)
objective = 1.5 - betas * 0.8

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(true_r, est_r, alpha=0.4, s=15, color='steelblue')
mn, mx = true_r.min(), true_r.max()
axes[0].plot([mn, mx], [mn, mx], 'r--')
axes[0].set(xlabel='True Reward', ylabel='Estimated', title=f'Reward Model (r={corr:.2f})')
axes[0].grid(True, alpha=0.3)

axes[1].plot(betas, objective, color='tomato', lw=2)
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].set(xlabel='KL Penalty β', ylabel='PPO Objective', title='Reward − β·KL')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/rlhf_demo.png')
print('Saved rlhf_demo.png')